### Load csv into pandas dataframe, parse date and time into single timestamp, filter out unimportant columns and low confidence records.

In [45]:
import pandas as pd
from pathlib import Path
import datetime

date = "2026-05-02"

file_path = Path.cwd().parent.joinpath(
    "data/raw", f"VIIRS_SNNP_NRT_world_14days_{date}.csv"
)

data = pd.read_csv(file_path)
# transform time in a string with actual utc mil time format
data["acq_time"] = data["acq_time"].apply(lambda x: f"{x:04d}")
data["timestamp"] = pd.to_datetime(
    data["acq_date"] + data["acq_time"], format="%Y-%m-%d%H%M", utc=True
)
data.drop(
    columns=["acq_date", "satellite", "instrument", "version", "acq_time"], inplace=True
)
data = data[data["confidence"] != "l"]

data.info()
data.head()


<class 'pandas.DataFrame'>
Index: 329115 entries, 0 to 379757
Data columns (total 10 columns):
 #   Column      Non-Null Count   Dtype              
---  ------      --------------   -----              
 0   latitude    329115 non-null  float64            
 1   longitude   329115 non-null  float64            
 2   bright_ti4  329115 non-null  float64            
 3   scan        329115 non-null  float64            
 4   track       329115 non-null  float64            
 5   confidence  329115 non-null  str                
 6   bright_ti5  329115 non-null  float64            
 7   frp         329115 non-null  float64            
 8   daynight    329115 non-null  str                
 9   timestamp   329115 non-null  datetime64[us, UTC]
dtypes: datetime64[us, UTC](1), float64(7), str(2)
memory usage: 27.6 MB


,latitude,longitude,bright_ti4,scan,track,confidence,bright_ti5,frp,daynight,timestamp
0,32.33215,44.09279,306.72,0.71,0.75,n,288.92,2.80,N,2026-05-01 00:01:00+00:00
1,32.88856,35.09304,296.63,0.44,0.46,n,281.80,1.02,N,2026-05-01 00:01:00+00:00
2,33.15357,44.78751,302.25,0.73,0.76,n,287.02,2.17,N,2026-05-01 00:01:00+00:00
3,33.15594,44.77987,316.28,0.73,0.76,n,288.05,2.17,N,2026-05-01 00:01:00+00:00
4,33.15751,44.78342,342.93,0.73,0.76,n,289.27,6.42,N,2026-05-01 00:01:00+00:00


### Load other data into geodataframe

In [54]:
import geopandas as gpd

fire_data = gpd.GeoDataFrame(
    data, geometry=gpd.points_from_xy(data.longitude, data.latitude), crs="EPSG:4326"
)

countries_area = pd.read_csv(Path.cwd().parent.joinpath("data/raw", "surface_area.csv"))

countries_boundaries = gpd.read_file(
    Path.cwd().parent.joinpath("data/raw", "country_polygons.geojson"), crs="EPSG:4326"
)

countries_boundaries.info()
countries_area.info()

/Users/nickf/sds210-project/.venv/lib/python3.14/site-packages/pyogrio/raw.py:200: RuntimeWarning: driver GeoJSON does not support open option CRS
  return ogr_read(


<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 258 entries, 0 to 257
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   name               258 non-null    str     
 1   ISO3166-1-Alpha-3  258 non-null    str     
 2   ISO3166-1-Alpha-2  258 non-null    str     
 3   geometry           258 non-null    geometry
dtypes: geometry(1), str(3)
memory usage: 8.2 KB
<class 'pandas.DataFrame'>
RangeIndex: 16410 entries, 0 to 16409
Data columns (total 41 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   STRUCTURE               16410 non-null  str    
 1   STRUCTURE_ID            16410 non-null  str    
 2   ACTION                  16410 non-null  str    
 3   FREQ                    16410 non-null  str    
 4   REF_AREA                16410 non-null  str    
 5   INDICATOR               16410 non-null  str    
 6   SEX               

### Filter out attributes in country area and polygons, cast into correct types

In [ ]:
countries_area = countries_area[["REF_AREA", "OBS_VALUE", "UNIT_TYPE"]]
countries_area["OBS_VALUE"] = countries_area["OBS_VALUE"].astype(float)
countries_boundaries = countries_boundaries.drop(columns="ISO3166-1-Alpha-2")